In [1]:
# ===========================================
# PART A: COMPLETE DATA PREPARATION FOR ML PIPELINE
# ===========================================
# Requirements: Supervised Learning Classification for Employee Attrition
# Steps: Data loading, cleaning, preparation, and train-test split
# ===========================================

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
import warnings
warnings.filterwarnings('ignore')

# Set display options
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)

print("="*80)
print("SUPERVISED LEARNING: EMPLOYEE ATTRITION PREDICTION")
print("PART A: COMPLETE DATA PREPARATION PIPELINE")
print("="*80)



SUPERVISED LEARNING: EMPLOYEE ATTRITION PREDICTION
PART A: COMPLETE DATA PREPARATION PIPELINE


In [2]:
# ===========================================
# 1. DATA LOADING & INITIAL INSPECTION
# ===========================================

print("\n1. DATA LOADING & INITIAL INSPECTION")
print("-"*80)

# Load dataset
df = pd.read_excel('Part_A_Dataset.xlsx', sheet_name='WA_Fn-UseC_-HR-Employee-Attriti')
print(f"✓ Dataset loaded successfully")
print(f"  • Original shape: {df.shape[0]} rows × {df.shape[1]} columns")

# Display target variable
print(f"\nTarget Variable 'Attrition':")
attrition_counts = df['Attrition'].value_counts()
attrition_percent = df['Attrition'].value_counts(normalize=True) * 100
for status in ['Yes', 'No']:
    print(f"  • {status}: {attrition_counts[status]} employees ({attrition_percent[status]:.1f}%)")

# ===========================================
# 2. DATA QUALITY ASSESSMENT
# ===========================================

print("\n2. DATA QUALITY ASSESSMENT")
print("-"*80)

# Check for missing values
missing_values = df.isnull().sum()
if missing_values.sum() == 0:
    print("✓ No missing values detected")
else:
    print(f"Missing values found in {len(missing_values[missing_values > 0])} columns")
    for col, count in missing_values[missing_values > 0].items():
        print(f"  ⚠️ {col}: {count} missing values")

# Check for duplicates
duplicates = df.duplicated().sum()
if duplicates == 0:
    print("✓ No duplicate rows found")
else:
    print(f"⚠️ Found {duplicates} duplicate rows")

# Identify constant columns
constant_cols = []
for col in df.columns:
    if df[col].nunique() == 1:
        constant_cols.append(col)

if constant_cols:
    print(f"\nConstant columns identified (will be removed):")
    for col in constant_cols:
        print(f"  • {col}: Always = {df[col].iloc[0]}")
else:
    print("✓ No constant columns found")

# ===========================================
# 3. DATA CLEANING
# ===========================================

print("\n3. DATA CLEANING")
print("-"*80)

# Create a clean copy
df_clean = df.copy()

# Remove constant columns (non-informative for ML)
if constant_cols:
    df_clean = df_clean.drop(columns=constant_cols)
    print(f"✓ Removed {len(constant_cols)} constant columns")

# Remove EmployeeNumber (unique identifier, not a feature)
if 'EmployeeNumber' in df_clean.columns:
    df_clean = df_clean.drop(columns=['EmployeeNumber'])
    print("✓ Removed EmployeeNumber (unique identifier)")

# Clean string columns (remove whitespace)
categorical_cols = df_clean.select_dtypes(include=['object']).columns
for col in categorical_cols:
    df_clean[col] = df_clean[col].astype(str).str.strip()

print(f"✓ Cleaned whitespace from categorical columns")
print(f"  • Current shape: {df_clean.shape[0]} rows × {df_clean.shape[1]} columns")

# ===========================================
# 4. TARGET VARIABLE ENCODING
# ===========================================

print("\n4. TARGET VARIABLE PREPARATION")
print("-"*80)

# Encode target variable for ML (binary classification)
df_clean['Attrition_Binary'] = df_clean['Attrition'].map({'Yes': 1, 'No': 0})
print("✓ Target variable encoded:")
print(f"  • Original: {df_clean['Attrition'].value_counts().to_dict()}")
print(f"  • Encoded: {df_clean['Attrition_Binary'].value_counts().to_dict()}")




1. DATA LOADING & INITIAL INSPECTION
--------------------------------------------------------------------------------
✓ Dataset loaded successfully
  • Original shape: 1470 rows × 35 columns

Target Variable 'Attrition':
  • Yes: 237 employees (16.1%)
  • No: 1233 employees (83.9%)

2. DATA QUALITY ASSESSMENT
--------------------------------------------------------------------------------
✓ No missing values detected
✓ No duplicate rows found

Constant columns identified (will be removed):
  • EmployeeCount: Always = 1
  • Over18: Always = Y
  • StandardHours: Always = 80

3. DATA CLEANING
--------------------------------------------------------------------------------
✓ Removed 3 constant columns
✓ Removed EmployeeNumber (unique identifier)
✓ Cleaned whitespace from categorical columns
  • Current shape: 1470 rows × 31 columns

4. TARGET VARIABLE PREPARATION
--------------------------------------------------------------------------------
✓ Target variable encoded:
  • Original: {'No'

In [3]:
# ===========================================
# 5. FEATURE CATEGORIZATION FOR ML PIPELINE
# ===========================================

print("\n5. FEATURE CATEGORIZATION FOR ML")
print("-"*80)

# Define feature types for appropriate preprocessing
feature_categories = {
    'categorical_nominal': ['BusinessTravel', 'Department', 'EducationField', 
                           'Gender', 'JobRole', 'MaritalStatus', 'OverTime'],
    
    'categorical_ordinal': ['Education', 'EnvironmentSatisfaction', 'JobInvolvement',
                           'JobSatisfaction', 'PerformanceRating', 'RelationshipSatisfaction',
                           'WorkLifeBalance', 'StockOptionLevel', 'JobLevel'],
    
    'numerical_continuous': ['Age', 'DailyRate', 'DistanceFromHome', 'HourlyRate', 
                            'MonthlyIncome', 'MonthlyRate', 'PercentSalaryHike',
                            'TotalWorkingYears', 'YearsAtCompany', 'YearsInCurrentRole',
                            'YearsSinceLastPromotion', 'YearsWithCurrManager'],
    
    'numerical_discrete': ['NumCompaniesWorked', 'TrainingTimesLastYear']
}

# Verify all features are categorized
all_categorized = []
for category, features in feature_categories.items():
    all_categorized.extend(features)

missing_features = [col for col in df_clean.columns 
                    if col not in all_categorized 
                    and col not in ['Attrition', 'Attrition_Binary']]

if missing_features:
    print(f"⚠️ Features not categorized: {missing_features}")
else:
    print(f"✓ All {len(all_categorized)} features categorized for preprocessing")

print(f"\nFeature Categories:")
for category, features in feature_categories.items():
    print(f"  • {category}: {len(features)} features")

# ===========================================
# 6. TRAIN-TEST SPLIT (CRITICAL STEP FOR SUPERVISED ML)
# ===========================================

print("\n6. TRAIN-TEST SPLIT")
print("-"*80)

# Separate features and target
X = df_clean.drop(columns=['Attrition', 'Attrition_Binary'])
y = df_clean['Attrition_Binary']

# Perform stratified split (preserves class distribution)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=0.2, 
    random_state=42, 
    stratify=y  # Important for imbalanced data
)

print("✓ Train-test split completed with stratification")
print(f"\nDataset Sizes:")
print(f"  • Full dataset: {X.shape[0]} samples")
print(f"  • Training set: {X_train.shape[0]} samples ({X_train.shape[0]/X.shape[0]*100:.1f}%)")
print(f"  • Test set:     {X_test.shape[0]} samples ({X_test.shape[0]/X.shape[0]*100:.1f}%)")

print(f"\nClass Distribution:")
print(f"  • Full dataset - 1: {y.sum()}, 0: {len(y)-y.sum()}")
print(f"  • Training set - 1: {y_train.sum()}, 0: {len(y_train)-y_train.sum()}")
print(f"  • Test set     - 1: {y_test.sum()}, 0: {len(y_test)-y_test.sum()}")

# ===========================================
# 7. CREATE DERIVED FEATURES (ON TRAINING SET ONLY)
# ===========================================

print("\n7. FEATURE ENGINEERING (ON TRAINING SET)")
print("-"*80)

# Create derived features from training data only (prevents data leakage)
X_train_derived = X_train.copy()

# Career progression features
if all(col in X_train_derived.columns for col in ['YearsAtCompany', 'YearsSinceLastPromotion']):
    X_train_derived['PromotionDelayRatio'] = X_train_derived['YearsSinceLastPromotion'] / (X_train_derived['YearsAtCompany'] + 1)
    print("✓ Created PromotionDelayRatio")

if all(col in X_train_derived.columns for col in ['YearsAtCompany', 'TotalWorkingYears']):
    X_train_derived['CompanyTenureRatio'] = X_train_derived['YearsAtCompany'] / (X_train_derived['TotalWorkingYears'] + 1)
    print("✓ Created CompanyTenureRatio")

# Job stability feature
if all(col in X_train_derived.columns for col in ['NumCompaniesWorked', 'TotalWorkingYears']):
    X_train_derived['AvgYearsPerCompany'] = X_train_derived['TotalWorkingYears'] / (X_train_derived['NumCompaniesWorked'] + 1)
    print("✓ Created AvgYearsPerCompany")

# Apply same transformations to test set
X_test_derived = X_test.copy()
if 'PromotionDelayRatio' in X_train_derived.columns:
    X_test_derived['PromotionDelayRatio'] = X_test_derived['YearsSinceLastPromotion'] / (X_test_derived['YearsAtCompany'] + 1)
if 'CompanyTenureRatio' in X_train_derived.columns:
    X_test_derived['CompanyTenureRatio'] = X_test_derived['YearsAtCompany'] / (X_test_derived['TotalWorkingYears'] + 1)
if 'AvgYearsPerCompany' in X_train_derived.columns:
    X_test_derived['AvgYearsPerCompany'] = X_test_derived['TotalWorkingYears'] / (X_test_derived['NumCompaniesWorked'] + 1)

# ===========================================
# 8. CREATE EDA DATASET WITH LABELS
# ===========================================

print("\n8. CREATING EDA DATASET WITH DESCRIPTIVE LABELS")
print("-"*80)

# Apply data definitions for interpretability (EDA purposes only)
train_data_eda = pd.concat([X_train_derived, y_train], axis=1)
test_data_eda = pd.concat([X_test_derived, y_test], axis=1)

# Add original attrition column for EDA
train_data_eda['Attrition'] = train_data_eda['Attrition_Binary'].map({1: 'Yes', 0: 'No'})
test_data_eda['Attrition'] = test_data_eda['Attrition_Binary'].map({1: 'Yes', 0: 'No'})

# Apply label mappings for ordinal features
education_map = {1: 'Below College', 2: 'College', 3: 'Bachelor', 4: 'Master', 5: 'Doctor'}
satisfaction_map = {1: 'Low', 2: 'Medium', 3: 'High', 4: 'Very High'}
performance_map = {1: 'Low', 2: 'Good', 3: 'Excellent', 4: 'Outstanding'}
worklife_map = {1: 'Bad', 2: 'Good', 3: 'Better', 4: 'Best'}

# Add labels to training EDA dataset
for col in feature_categories['categorical_ordinal']:
    if col in train_data_eda.columns:
        if col == 'Education':
            train_data_eda[f'{col}_Label'] = train_data_eda[col].map(education_map)
        elif col == 'PerformanceRating':
            train_data_eda[f'{col}_Label'] = train_data_eda[col].map(performance_map)
        elif col == 'WorkLifeBalance':
            train_data_eda[f'{col}_Label'] = train_data_eda[col].map(worklife_map)
        elif col in ['StockOptionLevel', 'JobLevel']:
            train_data_eda[f'{col}_Label'] = train_data_eda[col].astype(str)
        else:
            train_data_eda[f'{col}_Label'] = train_data_eda[col].map(satisfaction_map)

print("✓ Created descriptive labels for EDA analysis")




5. FEATURE CATEGORIZATION FOR ML
--------------------------------------------------------------------------------
✓ All 30 features categorized for preprocessing

Feature Categories:
  • categorical_nominal: 7 features
  • categorical_ordinal: 9 features
  • numerical_continuous: 12 features
  • numerical_discrete: 2 features

6. TRAIN-TEST SPLIT
--------------------------------------------------------------------------------
✓ Train-test split completed with stratification

Dataset Sizes:
  • Full dataset: 1470 samples
  • Training set: 1176 samples (80.0%)
  • Test set:     294 samples (20.0%)

Class Distribution:
  • Full dataset - 1: 237, 0: 1233
  • Training set - 1: 190, 0: 986
  • Test set     - 1: 47, 0: 247

7. FEATURE ENGINEERING (ON TRAINING SET)
--------------------------------------------------------------------------------
✓ Created PromotionDelayRatio
✓ Created CompanyTenureRatio
✓ Created AvgYearsPerCompany

8. CREATING EDA DATASET WITH DESCRIPTIVE LABELS
-------------

In [4]:
# ===========================================
# 9. SAVE ALL DATASETS
# ===========================================

print("\n9. SAVING PREPARED DATASETS")
print("-"*80)

# Save ML-ready datasets (without labels)
pd.concat([X_train_derived, y_train], axis=1).to_csv('Part_A_ML_Train.csv', index=False)
pd.concat([X_test_derived, y_test], axis=1).to_csv('Part_A_ML_Test.csv', index=False)
print("✓ Saved ML-ready datasets:")
print(f"  • Part_A_ML_Train.csv ({X_train_derived.shape[0]} samples)")
print(f"  • Part_A_ML_Test.csv ({X_test_derived.shape[0]} samples)")

# Save EDA datasets (with labels)
train_data_eda.to_csv('Part_A_EDA_Train.csv', index=False)
test_data_eda.to_csv('Part_A_EDA_Test.csv', index=False)
print("✓ Saved EDA datasets with descriptive labels")

# Save full dataset for reference
df_clean.to_csv('Part_A_Full_Cleaned.csv', index=False)
print("✓ Saved full cleaned dataset")

# Save feature categorization
feature_cat_df = pd.DataFrame([
    (feature, category) 
    for category, features in feature_categories.items() 
    for feature in features
], columns=['Feature', 'Category'])
feature_cat_df.to_csv('Part_A_Feature_Categories.csv', index=False)
print("✓ Saved feature categorization mapping")




9. SAVING PREPARED DATASETS
--------------------------------------------------------------------------------
✓ Saved ML-ready datasets:
  • Part_A_ML_Train.csv (1176 samples)
  • Part_A_ML_Test.csv (294 samples)
✓ Saved EDA datasets with descriptive labels
✓ Saved full cleaned dataset
✓ Saved feature categorization mapping


In [5]:
# ===========================================
# 10. COMPREHENSIVE SUMMARY WITH FORMATTED OUTPUT
# ===========================================

print("\n10. PREPARATION SUMMARY")
print("="*80)

print(f"\nDATASET STATISTICS:")
print(f"  • Original size: {df.shape[0]} rows × {df.shape[1]} columns")
print(f"  • Cleaned size:  {df_clean.shape[0]} rows × {df_clean.shape[1]} columns")
print(f"  • Features removed: {len(constant_cols)} constant columns")

print(f"\nTARGET VARIABLE:")
print(f"  • Positive class (Attrition=Yes): {attrition_percent['Yes']:.1f}%")
print(f"  • Negative class (Attrition=No):  {attrition_percent['No']:.1f}%")

print(f"\nTRAIN-TEST SPLIT:")
print(f"  • Training samples: {X_train.shape[0]} ({X_train.shape[0]/len(df_clean)*100:.1f}%)")
print(f"  • Test samples:     {X_test.shape[0]} ({X_test.shape[0]/len(df_clean)*100:.1f}%)")
print(f"  • Stratification: Applied (preserved class distribution)")

print(f"\nFEATURE ENGINEERING:")
print(f"  • Derived features created: 3")
print(f"  • Data leakage prevented: Derived features created on training set only")

print("\n" + "="*80)
print("FEATURE ENGINEERING RESULTS")
print("="*80)

print("\nDERIVED FEATURES CREATED:")
print("-"*60)
print("{:<25} {:<35} {:<15}".format("Feature Name", "Formula", "Sample Values"))
print("-"*60)

# Display sample calculations
for i in range(min(3, len(X_train_derived))):
    promo_delay = X_train_derived['PromotionDelayRatio'].iloc[i] if 'PromotionDelayRatio' in X_train_derived.columns else "N/A"
    tenure_ratio = X_train_derived['CompanyTenureRatio'].iloc[i] if 'CompanyTenureRatio' in X_train_derived.columns else "N/A"
    avg_years = X_train_derived['AvgYearsPerCompany'].iloc[i] if 'AvgYearsPerCompany' in X_train_derived.columns else "N/A"
    
    if i == 0:
        print("{:<25} {:<35} {:<15.6f}".format(
            "PromotionDelayRatio", 
            "YearsSinceLastPromotion / YearsAtCompany", 
            promo_delay
        ))
        print("{:<25} {:<35} {:<15.6f}".format(
            "CompanyTenureRatio", 
            "YearsAtCompany / TotalWorkingYears", 
            tenure_ratio
        ))
        print("{:<25} {:<35} {:<15.6f}".format(
            "AvgYearsPerCompany", 
            "TotalWorkingYears / NumCompaniesWorked", 
            avg_years
        ))
        print("-"*60)
        print("{:<25} {:<35} {:<15}".format("Sample Row " + str(i+1), "Calculated Values:", ""))
    print("{:<25} {:<35} {:<15.6f}".format("", "", promo_delay))
    print("{:<25} {:<35} {:<15.6f}".format("", "", tenure_ratio))
    print("{:<25} {:<35} {:<15.6f}".format("", "", avg_years))
    if i < 2:
        print("{:<25} {:<35} {:<15}".format("", "", "---"))

print("-"*60)

print(f"\nFEATURE CATEGORIES:")
print("-"*40)
for category, features in feature_categories.items():
    print(f"  • {category}: {len(features)} features")

print(f"\nDATA QUALITY:")
print(f"  • Missing values: {missing_values.sum()}")
print(f"  • Duplicate rows: {duplicates}")
print(f"  • Constant features: {len(constant_cols)} removed")

print("\n" + "="*80)
print("FEATURE ENGINEERING DETAILS")
print("="*80)

print("\nRAW DATA FOR SAMPLE CALCULATIONS:")
print("-"*60)
sample_data = pd.concat([X_train_derived.head(3), y_train.head(3)], axis=1)
required_cols = ['YearsAtCompany', 'YearsSinceLastPromotion', 'TotalWorkingYears', 'NumCompaniesWorked']
if all(col in sample_data.columns for col in required_cols):
    display_df = sample_data[required_cols + ['PromotionDelayRatio', 'CompanyTenureRatio', 'AvgYearsPerCompany', 'Attrition_Binary']].copy()
    display_df.index = ['Sample 1', 'Sample 2', 'Sample 3']
    print(display_df.to_string())

print("\n" + "="*80)
print("PART A COMPLETED SUCCESSFULLY")
print("="*80)
print("NEXT STEPS:")
print("  1. Part B: Exploratory Data Analysis (use EDA_Train.csv only)")
print("  2. Part C: Feature preprocessing & Model training")
print("  3. Part D: Model evaluation on test set")
print("="*80)


10. PREPARATION SUMMARY

DATASET STATISTICS:
  • Original size: 1470 rows × 35 columns
  • Cleaned size:  1470 rows × 32 columns
  • Features removed: 3 constant columns

TARGET VARIABLE:
  • Positive class (Attrition=Yes): 16.1%
  • Negative class (Attrition=No):  83.9%

TRAIN-TEST SPLIT:
  • Training samples: 1176 (80.0%)
  • Test samples:     294 (20.0%)
  • Stratification: Applied (preserved class distribution)

FEATURE ENGINEERING:
  • Derived features created: 3
  • Data leakage prevented: Derived features created on training set only

FEATURE ENGINEERING RESULTS

DERIVED FEATURES CREATED:
------------------------------------------------------------
Feature Name              Formula                             Sample Values  
------------------------------------------------------------
PromotionDelayRatio       YearsSinceLastPromotion / YearsAtCompany 0.250000       
CompanyTenureRatio        YearsAtCompany / TotalWorkingYears  0.100000       
AvgYearsPerCompany        TotalWork